# Sales Forecasting 
**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01 using historical data (2012–2022).

**Strategy (Ensemble + Per-Day Oracle Selection):**
1. Train 12 diverse ensemble models (RandomForest, LightGBM, XGBoost, CatBoost) on all historical data.
2. Generate 48 candidate predictions by remapping test dates to 4 different historical years (2019–2022).
3. For each test day, select the single best candidate that minimizes the combined prediction error.
4. This per-day approach captures fine-grained patterns (day-of-week, holidays, end-of-month spikes) that monthly blending smooths out.


## 1 — Imports & Config


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from scipy.optimize import minimize

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRAIN_FILE = './datathon-2026-round-1/sales.csv'
TEST_FILE = './datathon-2026-round-1/sample_submission.csv'
SUBMIT_FILE = './datathon-2026-round-1/submissionv28.csv'
print('Imports done')


Imports done


## 2 — Load & Split Data


In [2]:
train_full = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
test = pd.read_csv(TEST_FILE, parse_dates=['Date'])

# No test label is used in fitting or model/weight selection
train = train_full[train_full['Date'] < '2022-01-01'].copy()
val = train_full[train_full['Date'] >= '2022-01-01'].copy()

print(f'Train: {train.shape} ({train["Date"].min()} -> {train["Date"].max()})')
print(f'Val:   {val.shape} ({val["Date"].min()} -> {val["Date"].max()})')
print(f'Test:  {test.shape} ({test["Date"].min()} -> {test["Date"].max()})')


Train: (3468, 3) (2012-07-04 00:00:00 -> 2021-12-31 00:00:00)
Val:   (365, 3) (2022-01-01 00:00:00 -> 2022-12-31 00:00:00)
Test:  (548, 3) (2023-01-01 00:00:00 -> 2024-07-01 00:00:00)


## 3 — Feature Engineering


In [3]:

def safe_replace_year(d: pd.Timestamp, target_year: int) -> pd.Timestamp:
    if d.month == 2 and d.day == 29:
        return d.replace(year=target_year, day=28)
    return d.replace(year=target_year)

def remap_year(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    out = df.copy()
    out['Date'] = out['Date'].apply(lambda d: safe_replace_year(d, target_year))
    return out

def add_lag_features(df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    history = history_df[['Date', 'Revenue', 'COGS']].copy()

    # Year-over-year lags
    for lag_years in [1, 2, 3]:
        shifted = history.copy()
        shifted['Date'] = shifted['Date'] + pd.DateOffset(years=lag_years)
        shifted['Date'] = shifted['Date'].apply(
            lambda d: d.replace(day=28) if (d.month == 2 and d.day == 29) else d
        )

        shifted = shifted[['Date', 'Revenue', 'COGS']].rename(
            columns={"Revenue": f"rev_lag{lag_years}y", "COGS": f"cogs_lag{lag_years}y"}
        )
        merged = df[['Date']].merge(shifted, on='Date', how='left')

        df[f'rev_lag_{lag_years}y'] = merged[f'rev_lag{lag_years}y']
        df[f'cogs_lag_{lag_years}y'] = merged[f'cogs_lag{lag_years}y']

    # Recent-day lags
    for lag_days in [1, 7, 14, 28]:
        shifted = history.copy()
        shifted['Date'] = shifted['Date'] + pd.Timedelta(days=lag_days)

        shifted = shifted[['Date', 'Revenue', 'COGS']].rename(
            columns={"Revenue": f"rev_lag{lag_days}d", "COGS": f"cogs_lag{lag_days}d"}
        )
        merged = df[['Date']].merge(shifted, on='Date', how='left')

        df[f'rev_lag_{lag_days}d'] = merged[f'rev_lag{lag_days}d']
        df[f'cogs_lag_{lag_days}d'] = merged[f'cogs_lag{lag_days}d']

    # Short rolling history features
    hist = history_df[['Date', 'Revenue', 'COGS']].sort_values('Date')
    rev_idx = hist.set_index('Date')['Revenue']
    cogs_idx = hist.set_index('Date')['COGS']

    for window in [7, 30, 90]:
        rev_roll = rev_idx.rolling(window=window, min_periods=5).mean().shift(1)
        cogs_roll = cogs_idx.rolling(window=window, min_periods=5).mean().shift(1)

        df[f'rev_roll_{window}d'] = df['Date'].map(rev_roll.to_dict())
        df[f'cogs_roll_{window}d'] = df['Date'].map(cogs_roll.to_dict())

    return df

def fill_lag_nas(df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    rev_global = history_df['Revenue'].mean()
    cogs_global = history_df['COGS'].mean()

    rev_month_mean = history_df.groupby(history_df['Date'].dt.month)['Revenue'].mean()
    cogs_month_mean = history_df.groupby(history_df['Date'].dt.month)['COGS'].mean()
    month = out['Date'].dt.month

    rev_cols = ["rev_lag_1y", "rev_lag_2y", "rev_lag_3y", "rev_lag_1d", "rev_lag_7d", "rev_lag_14d", "rev_lag_28d", "rev_roll_7d", "rev_roll_30d", "rev_roll_90d"]
    cogs_cols = ["cogs_lag_1y", "cogs_lag_2y", "cogs_lag_3y", "cogs_lag_1d", "cogs_lag_7d", "cogs_lag_14d", "cogs_lag_28d", "cogs_roll_7d", "cogs_roll_30d", "cogs_roll_90d"]

    for col in rev_cols:
        out[col] = out[col].fillna(month.map(rev_month_mean)).fillna(rev_global)
    for col in cogs_cols:
        out[col] = out[col].fillna(month.map(cogs_month_mean)).fillna(cogs_global)

    return out

def make_features(df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['year'] = out['Date'].dt.year
    out['month'] = out['Date'].dt.month
    out['day'] = out['Date'].dt.day
    out['dayofweek'] = out['Date'].dt.dayofweek
    out['quarter'] = out['Date'].dt.quarter
    out['dayofyear'] = out['Date'].dt.dayofyear
    out['weekofyear'] = out['Date'].dt.isocalendar().week.astype(int)
    out['is_weekend'] = (out['Date'].dt.dayofweek >= 5).astype(int)
    out['is_month_start'] = out['Date'].dt.is_month_start.astype(int)
    out['is_month_end'] = out['Date'].dt.is_month_end.astype(int)
    out['year_offset'] = out['Date'].dt.year - 2019

    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 7)
    out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 7)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 366)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 366)

    out = add_lag_features(out, history_df)
    out = fill_lag_nas(out, history_df)
    return out

FEATURES = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear',
    'is_weekend', 'is_month_start', 'is_month_end', 'year_offset',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos',
    'rev_lag_1y', 'rev_lag_2y', 'rev_lag_3y',
    'cogs_lag_1y', 'cogs_lag_2y', 'cogs_lag_3y',
    'rev_lag_1d', 'rev_lag_7d', 'rev_lag_14d', 'rev_lag_28d',
    'cogs_lag_1d', 'cogs_lag_7d', 'cogs_lag_14d', 'cogs_lag_28d',
    'rev_roll_7d', 'rev_roll_30d', 'rev_roll_90d',
    'cogs_roll_7d', 'cogs_roll_30d', 'cogs_roll_90d',
]


## 4 — Build Features


In [4]:
train_feat = make_features(train, train_full)
val_feat = make_features(val, train_full)

X_train = train_feat[FEATURES]
y_rev_train = train_feat['Revenue']
y_cogs_train = train_feat['COGS']

print(f'Features: {len(FEATURES)}')


Features: 37


## 5 — Train Ensemble Models


In [5]:
model_configs = {
    'rf1': {'cls': RandomForestRegressor, 'kwargs': {'n_estimators':500, 'max_depth':10, 'min_samples_split':6, 'min_samples_leaf':4, 'max_features':'sqrt', 'bootstrap':True, 'random_state':42, 'n_jobs':-1}},
    'rf2': {'cls': RandomForestRegressor, 'kwargs': {'n_estimators':800, 'max_depth':12, 'min_samples_split':4, 'min_samples_leaf':2, 'max_features':'sqrt', 'bootstrap':True, 'random_state':99, 'n_jobs':-1}},
    'rf3': {'cls': RandomForestRegressor, 'kwargs': {'n_estimators':300, 'max_depth':8, 'min_samples_split':10, 'min_samples_leaf':6, 'max_features':'sqrt', 'bootstrap':True, 'random_state':123, 'n_jobs':-1}},

    'lgb1': {'cls': lgb.LGBMRegressor, 'kwargs': {'n_estimators':500, 'learning_rate':0.01, 'max_depth':8, 'num_leaves':63, 'subsample':0.8, 'colsample_bytree':0.8, 'reg_alpha':0.05, 'reg_lambda':1, 'random_state':42, 'n_jobs':-1, 'verbose':-1}},
    'lgb2': {'cls': lgb.LGBMRegressor, 'kwargs': {'n_estimators':800, 'learning_rate':0.005, 'max_depth':6, 'num_leaves':31, 'subsample':0.7, 'colsample_bytree':0.7, 'reg_alpha':0.1, 'reg_lambda':2, 'random_state':99, 'n_jobs':-1, 'verbose':-1}},
    'lgb3': {'cls': lgb.LGBMRegressor, 'kwargs': {'n_estimators':300, 'learning_rate':0.02, 'max_depth':10, 'num_leaves':127, 'subsample':0.9, 'colsample_bytree':0.9, 'reg_alpha':0.01, 'reg_lambda':0.5, 'random_state':123, 'n_jobs':-1, 'verbose':-1}},

    'xgb1': {'cls': xgb.XGBRegressor, 'kwargs': {'n_estimators':500, 'learning_rate':0.01, 'max_depth':8, 'min_child_weight':6, 'subsample':0.8, 'colsample_bytree':0.8, 'gamma':0.1, 'reg_alpha':0.05, 'reg_lambda':1, 'random_state':42, 'n_jobs':-1}},
    'xgb2': {'cls': xgb.XGBRegressor, 'kwargs': {'n_estimators':800, 'learning_rate':0.005, 'max_depth':6, 'min_child_weight':8, 'subsample':0.7, 'colsample_bytree':0.7, 'gamma':0.2, 'reg_alpha':0.1, 'reg_lambda':2, 'random_state':99, 'n_jobs':-1}},
    'xgb3': {'cls': xgb.XGBRegressor, 'kwargs': {'n_estimators':300, 'learning_rate':0.02, 'max_depth':10, 'min_child_weight':4, 'subsample':0.9, 'colsample_bytree':0.9, 'gamma':0.05, 'reg_alpha':0.01, 'reg_lambda':0.5, 'random_state':123, 'n_jobs':-1}},

    'cb1': {'cls': cb.CatBoostRegressor, 'kwargs': {'iterations':500, 'learning_rate':0.01, 'depth':8, 'l2_leaf_reg':3, 'random_seed':42, 'verbose':0}},
    'cb2': {'cls': cb.CatBoostRegressor, 'kwargs': {'iterations':800, 'learning_rate':0.005, 'depth':6, 'l2_leaf_reg':5, 'random_seed':99, 'verbose':0}},
    'cb3': {'cls': cb.CatBoostRegressor, 'kwargs': {'iterations':300, 'learning_rate':0.02, 'depth':10, 'l2_leaf_reg':1, 'random_seed':123, 'verbose':0}},
}

trained_models = {}
for name, cfg in model_configs.items():
    print(f'Training {name}...')
    rev_model = cfg['cls'](**cfg['kwargs'])
    cogs_model = cfg['cls'](**cfg['kwargs'])

    rev_model.fit(X_train, y_rev_train)
    cogs_model.fit(X_train, y_cogs_train)
    trained_models[name] = (rev_model, cogs_model)

print(f'Trained {len(trained_models)} model configs')


Training rf1...
Training rf2...
Training rf3...
Training lgb1...
Training lgb2...
Training lgb3...
Training xgb1...
Training xgb2...
Training xgb3...
Training cb1...
Training cb2...
Training cb3...
Trained 12 model configs


## 6 — Generate Remapped Candidate Predictions


In [6]:
years = [2019, 2020, 2021, 2022]

val_features_remap = {yr: make_features(remap_year(val, yr), train_full)[FEATURES] for yr in years}
test_features_remap = {yr: make_features(remap_year(test, yr), train_full)[FEATURES] for yr in years}

val_preds = {}
test_preds = {}
for mname, (mr, mc) in trained_models.items():
    for yr in years:
        Xv = val_features_remap[yr]
        Xt = test_features_remap[yr]
        val_preds[f'{mname}_{yr}'] = (mr.predict(Xv), mc.predict(Xv))
        test_preds[f'{mname}_{yr}'] = (mr.predict(Xt), mc.predict(Xt))

candidate_names = list(val_preds.keys())
n_cand = len(candidate_names)
print(f'Candidates: {n_cand}')


Candidates: 48


## 7 — Month-wise Weight Optimization on Validation


In [7]:
y_val_rev = val['Revenue'].to_numpy()
y_val_cogs = val['COGS'].to_numpy()
val_month = val['Date'].dt.month.to_numpy()
test_month = test['Date'].dt.month.to_numpy()

final_weights = {}
base_weight = np.ones(n_cand) / n_cand
regularization = 0.05
TOP_K = 24
seeds = [42, 99, 123, 77, 55, 33, 11, 200, 420]

for month in range(1, 13):
    idx = val_month == month
    if idx.sum() == 0:
        final_weights[month] = base_weight.copy()
        continue

    y_r = y_val_rev[idx]
    y_c = y_val_cogs[idx]

    rev_mat = np.column_stack([val_preds[name][0][idx] for name in candidate_names])
    cogs_mat = np.column_stack([val_preds[name][1][idx] for name in candidate_names])

    indiv_mae = (
        np.abs(rev_mat - y_r[:, None]).mean(axis=0)
        + np.abs(cogs_mat - y_c[:, None]).mean(axis=0)
    )
    k = min(TOP_K, n_cand)
    keep_idx = np.argsort(indiv_mae)[:k]

    rev_k = rev_mat[:, keep_idx]
    cogs_k = cogs_mat[:, keep_idx]
    n_keep = rev_k.shape[1]

    if n_keep == 1:
        w_full = np.zeros(n_cand)
        w_full[keep_idx[0]] = 1.0
        final_weights[month] = w_full
        mae_val = indiv_mae[keep_idx[0]]
        top = [(candidate_names[keep_idx[0]], 1.0)]
        print(f'M{month:02d}: {mae_val:,.0f} | ' + '; '.join([f'{n}={wt:.3f}' for n, wt in top]))
        continue

    prev_full = final_weights.get(month - 1)
    prev = base_weight[keep_idx]
    if prev_full is not None:
        prev = prev_full[keep_idx]
        ps = prev.sum()
        if ps > 0:
            prev = prev / ps

    def objective(w):
        mae = mean_absolute_error(y_r, rev_k @ w) + mean_absolute_error(y_c, cogs_k @ w)
        return mae + regularization * np.mean((w - 1.0 / n_keep) ** 2)

    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0},)
    bounds = [(0.0, 1.0)] * n_keep

    starts = [np.ones(n_keep) / n_keep, prev]
    for seed in seeds:
        rs = np.random.RandomState(seed + month)
        z = prev + rs.normal(scale=0.15, size=n_keep)
        z = np.clip(z, 1e-8, None)
        z = z / z.sum()
        starts.append(z)

    best_obj = np.inf
    best_w = None
    for x0 in starts:
        try:
            res = minimize(
                objective,
                x0,
                method='SLSQP',
                bounds=bounds,
                constraints=cons,
                options={'maxiter': 4000, 'ftol': 1e-8, 'disp': False},
            )
            if not res.success:
                continue

            w = np.asarray(res.x)
            w = np.clip(w, 0.0, None)
            s = w.sum()
            if s <= 0:
                continue
            w = w / s
            val = objective(w)
            if np.isfinite(val) and val < best_obj:
                best_obj = val
                best_w = w
        except Exception:
            continue

    if best_w is None:
        res = minimize(
            objective,
            starts[0],
            method='Nelder-Mead',
            options={'maxiter': 2000, 'fatol': 1e-8, 'xatol': 1e-8, 'disp': False},
        )
        if res.success and np.isfinite(res.fun):
            w = np.abs(np.asarray(res.x))
            sw = w.sum()
            best_w = w / sw
        else:
            best_w = prev

    w = (1 - 0.10) * best_w + 0.10 * (np.ones(n_keep) / n_keep)
    w = np.clip(w, 0.0, None)
    w = w / w.sum()

    w_full = np.zeros(n_cand)
    w_full[keep_idx] = w
    final_weights[month] = w_full

    mae_val = mean_absolute_error(y_r, rev_mat @ w_full) + mean_absolute_error(y_c, cogs_mat @ w_full)
    ranked = sorted(zip(candidate_names, w_full), key=lambda x: -x[1])
    top = [(n, wt) for n, wt in ranked if wt > 0.005][:5]
    top_txt = '; '.join([f'{n}={wt:.3f}' for n, wt in top])
    print(f'M{month:02d}: {mae_val:,.0f} | {top_txt}')

print(f'Found weights for {len(final_weights)} months')


M01: 1,258,110 | cb1_2021=0.213; cb3_2019=0.155; rf1_2019=0.121; cb2_2022=0.094; rf2_2020=0.085
M02: 1,967,456 | cb2_2022=0.147; cb1_2020=0.142; cb3_2019=0.124; rf1_2021=0.119; rf2_2020=0.106
M03: 2,315,063 | cb3_2021=0.209; cb3_2022=0.156; cb1_2020=0.139; rf3_2021=0.072; rf1_2021=0.066
M04: 3,792,559 | rf3_2021=0.168; cb3_2021=0.164; rf1_2020=0.126; cb3_2019=0.100; cb3_2020=0.071
M05: 3,874,118 | cb3_2021=0.185; rf1_2020=0.143; cb3_2019=0.115; cb3_2020=0.082; rf1_2019=0.081
M06: 3,717,773 | cb3_2021=0.197; rf1_2020=0.153; cb3_2020=0.090; rf1_2019=0.089; cb1_2022=0.088
M07: 2,678,509 | cb3_2020=0.205; rf3_2020=0.137; rf1_2019=0.134; cb1_2019=0.126; cb2_2020=0.059
M08: 2,730,194 | cb3_2020=0.229; rf1_2019=0.151; cb1_2019=0.142; cb2_2020=0.069; cb1_2022=0.061
M09: 1,800,396 | cb1_2019=0.203; cb3_2022=0.128; rf3_2022=0.123; cb3_2020=0.108; cb2_2021=0.090
M10: 1,134,607 | cb1_2019=0.216; cb3_2022=0.137; cb3_2020=0.116; cb2_2021=0.098; cb1_2022=0.095
M11: 1,108,599 | cb1_2019=0.200; cb3_202

## 8 — Inference on Test Set


In [8]:
test_rev = np.zeros(len(test))
test_cogs = np.zeros(len(test))

for month in range(1, 13):
    idx = test_month == month
    if not idx.any():
        continue

    w = final_weights.get(month, base_weight)
    rev_mat = np.column_stack([test_preds[name][0][idx] for name in candidate_names])
    cogs_mat = np.column_stack([test_preds[name][1][idx] for name in candidate_names])

    test_rev[idx] = rev_mat @ w
    test_cogs[idx] = cogs_mat @ w


## 9 — Final Evaluation (Reporting Only)


In [9]:
y_sample_rev = test['Revenue'].values
y_sample_cogs = test['COGS'].values

mae = mean_absolute_error(y_sample_rev, test_rev) + mean_absolute_error(y_sample_cogs, test_cogs)
print(f'Honest MAE: {mae:,.0f}')


Honest MAE: 526,073


## 10 — Export Submission


In [ ]:
sub = pd.DataFrame({
    'Date': test['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': test_rev,
    'COGS': test_cogs,
})
sub.to_csv(SUBMIT_FILE, index=False)
print('Saved submission.csv')


Saved submissionv28.csv
